In [ ]:
# p.142-146 本と同じコード Google Colabでは動かない

import cv2

# カメラの初期化
cap = cv2.VideoCapture(0)

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()

# 独自クラス ColabCap を作った. これを cap 名でインスタンス化し本 p.146以降と同等のコードにする

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2

class ColabCap:
  # 本の cap = cv2.VideoCapture(0) の代用クラス
  # TODO: 動画撮影用メソッド追加 https://qiita.com/sueasen/items/abf2d27c888c4d6d268d
  # サイズやフレームレートも変更可 https://qiita.com/yusuke84/items/35750017a6b12199aa39
  # https://developer.mozilla.org/ja/docs/Web/API/Media_Capture_and_Streams_API/Constraints

  _js = '''
    let video = document.createElement('video');
    let canvas = document.createElement('canvas');
    let stream = null;

    async function createDom() {
      if (stream) return;
      stream = await navigator.mediaDevices.getUserMedia({ video: true });
      video.srcObject = stream;
      await video.play();
    }

    async function removeDom() {
      await stream.getVideoTracks()[0].stop();
      video = null;
      stream = null;
      canvas = null;
    }

    async function cap(quality, waitSec) {
      if (!stream) await createDom();
      await new Promise((resolve) => setTimeout(resolve, waitSec * 10**3));
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg', quality);
    }
  '''

  def __init__(self, quality=0.8, first_wait_sec=0.25):
    self.quality = quality
    self.first_wait_sec = first_wait_sec
    display(Javascript(ColabCap._js))

  def read(self):
    try:
      data = eval_js(f'cap({ self.quality }, { self.first_wait_sec })')
      self.first_wait_sec = 0
      image_bytes = b64decode(data.split(',')[1])
      jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
      return True, cv2.imdecode(jpg_as_np, flags=1)
    except Exception as err:
      print(str(err))
      return False, None

  def release(self):
    eval_js('removeDom()')


In [ ]:
# 独自クラスColabCapを使い、本 p.146 (5-1-2) と同じ結果を得る (静止画撮影・保存)

# 実行前に1回ColabCapを定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.4.17 Chrome 146.0, Firefox 149.0

# カメラの初期化
# cap = cv2.VideoCapture(0) # 本
cap = ColabCap() # Colab版

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()


In [ ]:
# 独自クラスColabCapを使い、本 p.148 (5-1-3) と同じ結果を得る (静止画撮影・保存・表示)
# 画像表示はColab用ライブラリを使う

# 実行前に1回 ColabCap を定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.3.27 Chrome 146.0, Firefox 148.0

from google.colab.patches import cv2_imshow

# カメラの初期化
cap = ColabCap()

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()

# 結果表示
cv2_imshow(frame)


In [ ]:
# 独自クラスColabCapを使い、本 p.149〜153 (5-2) と似た結果を得る (動画撮影・保存)
# この方法では滑らかな動画にならない. 1枚1枚をブラウザで取得しColabへ送っているため
# 後で動画用のメソッド作ったらそのバージョンを追加 [TODO]
# 動画表示も遅いので省略

# 実行前に1回 ColabCap を定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.3.27 Chrome 146.0, Firefox 148.0

import cv2
import time

# カメラの初期化
cap = ColabCap()

# 撮影条件
frame_rate = 10 # Colabでは10fps位が限界. sleepなしで限界を確かめ設定するのが現実的
duration = 10
interval = 1 / frame_rate
frame_count = int(duration / interval)

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie.mp4', fourcc, frame_rate, (640, 480))

# 動画撮影
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break
  out.write(frame)

  # 再生すると極端に遅くなる (出力欄をつど消去するため)
  # cv2_imshow(frame)
  # output.clear()

  # time.sleep(interval)
  # Colabではフレームレートを高くできないので不要

cap.release()
out.release()


In [ ]:
# 独自クラスColabCapを使い、本 p.154〜157 (5-3) と似た結果を得る (タイムラプス動画撮影・保存)

# 実行前に1回 ColabCap を定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.3.27 Chrome 146.0, Firefox 148.0

import cv2
import time
from google.colab.patches import cv2_imshow
from google.colab import output

# カメラの初期化
cap = ColabCap()

# 撮影条件
frame_rate = 10
duration = 60
interval = 2
frame_count = int(duration / interval)

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie_timelapse.mp4', fourcc, frame_rate, (640, 480))

# 動画撮影
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break
  out.write(frame)

  cv2_imshow(frame)

  time.sleep(interval) # 実際はintervalより少し長めに待機してしまう. ブラウザの処理と通信が入るため
  output.clear()

cap.release()
out.release()


In [ ]:
# 5-4-1 Colab版
import cv2
from google.colab.patches import cv2_imshow

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像の表示
cv2_imshow(img)

In [ ]:
# 5-4-2 Colab版
from google.colab.patches import cv2_imshow
import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア調にするカラー変換行列
  # 注：0.xxxの先頭ゼロは略せる 工学系のよくある記法
  # 注：インデント浅くした

  sepia_filter = np.array([
    [.272, .534, .131],
    [.349, .686, .168],
    [.393, .769, .189]
  ])
  applied_img = cv2.transform(img, sepia_filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_color_tone(img)

# 画像の表示
cv2_imshow(applied_img)

# 画像の保存
cv2.imwrite('img_out.jpg', applied_img)


In [ ]:
# 5-4-2 Colab版
from google.colab.patches import cv2_imshow
import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア以外の変換の例 本p.166 ⑥全体的に明るくする
  filter = np.array([
    [3, 0, 0],
    [0, 3, 0],
    [0, 0, 3]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_color_tone(img)

# 画像の表示
cv2_imshow(applied_img)

# 画像の保存
cv2.imwrite('img_out.jpg', applied_img)


In [ ]:
# 5-4-4 Colab版
from google.colab.patches import cv2_imshow
import cv2

def apply_edges(img):
  """エッジを検出して元画像に重ね描きする関数"""

  # グレースケールに変換
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # エッジを検出
  edges = cv2.Canny(gray, 100, 200)

  # エッジを黒色で描画
  img[edges == 255] = (0, 0, 0)

  return img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_edges(img)

# 画像の表示
cv2_imshow(applied_img)


In [ ]:
# 5-4-5 Colab版
import numpy as np
from google.colab.patches import cv2_imshow
import cv2

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_edges(img)

# 画像の表示
cv2_imshow(applied_img)


In [ ]:
# 5-4-6 Colab版
from google.colab.patches import cv2_imshow
import cv2

def apply_blur(img):
  """画像全体にぼかし効果を追加する関数"""
  kernel = (15, 15)
  return cv2.GaussianBlur(img, kernel, 0)

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_blur(img)

# 画像の表示
cv2_imshow(applied_img)


In [ ]:
# 5-5-1 一部簡略化した全体版
import cv2

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア以外の変換の例 本p.166 ⑥全体的に明るくする
  filter = np.array([
    [3, 0, 0],
    [0, 3, 0],
    [0, 0, 3]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

# 動画ファイルの読み込み
cap = cv2.VideoCapture('movie.mp4')

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# 注：本の55行目 interval はどこからも呼ばれず不要

# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie_edited.mp4', fourcc, frame_rate, (w, h))

# 動画再生
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break

  # 画像処理を実行
  out.write(apply_color_tone(img))

# ファイルの解放
cap.release()
out.release()


In [ ]:
# 5-5-2 一部簡略化した全体版
import cv2

def apply_color_tone(img):
  filter = np.array([
    [5, 0, 0],
    [0, 5, 0],
    [0, 0, 5]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  return np.clip(applied_img, 0, 255).astype(np.uint8)

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# 動画ファイルの読み込み
cap = cv2.VideoCapture('movie.mp4')

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie_edited_2.mp4', fourcc, frame_rate, (w, h))

# 動画再生
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break

  # 画像処理を実行
  applied_image = apply_color_tone(img)
  applied_image = apply_edges(applied_image)
  out.write(applied_image)

# ファイルの解放
cap.release()
out.release()
